Reading from Bronze

INIT


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, size, split
from pyspark.sql.functions import year, month, dayofmonth
from pyspark.sql.functions import to_date
from pyspark.sql.functions import col, sum as spark_sum

In [0]:
df = spark.table("workspace.bronze.sales_details")

Data Transformation

In [0]:
df.display()

In [0]:
cols_to_keep = ["ROW_ID","Order_ID","Order_Date","Ship_Date","Ship_mode","Customer_ID","Customer_Name","Segment","Country","City","execution_datetime","source_file"]

df = df.select(*cols_to_keep)

Trimming


In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name, trim(col(field.name)))



In [0]:
df = df.withColumn("order_date", to_date("order_date"))

In [0]:
RENAME_MAP = {
        "ROW_ID": "row_id",
        "Order_ID": "order_id",
        "Order_Date": "order_date",
        "Ship_Date": "ship_date",
        "Ship_Mode": "ship_mode",
        "Customer_ID": "customer_id",
        "Customer_Name": "customer_name",
        "Segment": "segment",
        "Country": "country",
        "City": "city",
        "State": "state"
    }

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()


In [0]:
name_split = split(col("customer_name"), " ")

df = (
    df
    .withColumn("customer_first_name", name_split[0])
    .withColumn("customer_last_name", name_split[size(name_split)-1])
    .drop("customer_name")
)

In [0]:
df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

In [0]:
df = df.dropDuplicates(["order_id"])

In [0]:
df = df.filter(
    (col("order_date").isNotNull()) &
    (col("ship_date").isNotNull()) &
    (col("ship_date") >= col("order_date"))
)workspace.silver.orders_partitioned

In [0]:
df = (
    df
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .withColumn("day", dayofmonth("order_date"))
)

Write into silver table

In [0]:
df.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year", "month", "day") \
    .saveAsTable("silver.orders")

In [0]:
spark.sql("SELECT * FROM silver.orders").show()